In [587]:
import numpy as np

def randList(size):
    result = []
    for i in range(size):
        result.append(np.random.rand())
    return result

def vectorAddition(list1,list2):
    result = []
    for i in range(len(list1)):
        result.append(list1[i] + list2[i])
    return result

def ReLU(num):
    if num <= 0:
        return 0
    else:
        return num

def tanh(num):
    return np.tanh(num)

def activations(name):
    if name == "ReLU":
        return ReLU
    elif name == "tanh":
        return tanh

def min_max(data):
    result = []
    x_min = min(data)
    x_max = max(data)
    for x in data:
        f_x = (x - x_min)/(x_max - x_min)
        result.append(f_x)
    return result

def standard(data):
    result = []
    mean = np.array(data).mean()
    std = np.array(data).std()
    for x in data:
        f_x = (x - mean)/std
        result.append(f_x)
    return result

def normalizations(name):
    if name == "min_max":
        return min_max
    elif name == "standard":
        return standard

In [479]:
class Node:
    def __init__(self, value):
        self.value = value
    def __init__(self):
        self.value = 1
        
    def getValue(self):
        return self.value
        
    def update(self, value):
        self.value = value

In [480]:
class nodeLayer:
    def __init__(self):
        self.nodes = []

    def __init__(self, size):
        self.nodes = []
        for i in range(0,size):
            temp = Node()
            self.nodes.append(temp)
            temp = 0
        
    def addNode(self, node: Node):
        self.nodes.append(node)

    def addNodes(self, many: list):
        for node in many:
            self.nodes.append(node)

    def updateNodes(self, func):
        for obj in self.nodes:
            obj = obj.update(func(obj.getValue))

    def getList(self):
        result = []
        for obj in self.nodes:
            result.append(obj.getValue())
        return result

In [487]:
class activationLayer:
    def __init__(self, name: str):
        self.func = activations(name)
        
    def activate(self, num):
        return self.func(num)

    def isActivation(self):
        return True

    def isNormalization(self):
        return False

class normalizationLayer:
    def __init__(self, name: str):
        self.func = normalizations(name)

    def normalize(self, num):
        return self.func(num)

    def isNormalization(self):
        return True

    def isActivation(self):
        return False


class Layer:
    def __init__(self, inputSize, outputSize = None):
        if outputSize == None:
            self.input, self.output = inputSize, inputSize
            self.isTail = True
        else:
            self.input, self.output = inputSize, outputSize
            
    def getInput(self):
        return self.input
    def getSize(self):
        return self.input
    def isActivation(self):
        return False
    def isNormalization(self):
        return False
    def getOutput(self):
        return self.output
    def tail(self):
        return self.isTail
        

In [516]:
class Vector:
    def __init__(self, data):
        self.values = []
        if isinstance(data, int):
            for i in range(0,data):
                self.values.append(np.random.rand()*2 - 1)
        elif isinstance(data, list):
            self.values = data
        else:
            raise TypeError("weh")
    def getValues(self):
        return self.values
    def getLength(self):
        return len(self.values)

class randMatrix:
    def __init__(self, m, n):
        self.val = []
        self.rows = m
        self.cols = n
        for i in range(0,m):
            temp = []
            for j in range(0, n):
                temp.append(np.random.rand()*2 - 1)
            self.val.append(temp)
    def getValues(self):
        return self.val

    def vectorMultiply(self, vec):
        if len(vec) != self.cols:
            raise ValueError("Vector size mismatch")
        else:
            result = []
            for row in self.val:
                temp = 0
                for i in range(len(row)):
                    temp += row[i] * vec[i]
                result.append(temp)
            return result


In [605]:
class Model:
    def __init__(self):
        self.sequence = []
        self.inputTrace = []
        self.biases = []
        self.weights = []
        
    def setLayers(self, layers):
        for i in range(0,len(layers)):
            curLayer = layers[i]
            if(curLayer.isActivation()):
                self.sequence.append(["Activation", curLayer])
                pass
            elif(curLayer.isNormalization()):
                self.sequence.append(["Normalization", curLayer])
                pass
            else:

                cols = curLayer.getInput()
                rows = curLayer.getOutput()

                self.sequence.append([
                    "Weights",
                    randMatrix(rows,cols).getValues(),
                    randMatrix(rows,cols)
                ])

                self.sequence.append([
                    "Biases",
                    randList(rows),
                ])
            
                
                
    def getSequence(self):
        return self.sequence
    def getLayers(self):
        for i in range(0,len(self.sequence)):
            print(f"{i} " + self.sequence[i][0])
        #for layer in self.sequence:
         #   print(layer[0])
    def getTrace(self):
        return self.inputTrace

    def evaluateInput(self, vector):
        #if len(vector) != self.sequence[0].getSize():
        #    raise ValueError("size mismatch")
        #else:
        result = vector
        for i in range(len(self.sequence)):
            self.inputTrace.append([f"{i}th in sequence",result])
            if self.sequence[i][0] == "Weights":
                result = self.sequence[i][2].vectorMultiply(result)
            elif self.sequence[i][0] == "Biases":
                result = vectorAddition(result,self.sequence[i][1])
                
            elif self.sequence[i][0] == "Activation":
                for j in range(0, len(result)):
                    result[j] = self.sequence[i][1].activate(result[j])
                    
            elif self.sequence[i][0] == "Normalization":
                result = self.sequence[i][1].normalize(result)
            else:
                print("what?")
                    #result = self.sequence[i][1]
                #result = self.weights[i][2].vectorMultiply(result)
                #result = vectorAddition(result,self.biases[i][1])
            
        print(result)
        return result

In [606]:
test = Model()
test.setLayers([
    normalizationLayer("min_max"),
    Layer(3,3),
    activationLayer("ReLU"),
    Layer(3,2),
    activationLayer("ReLU"),
    Layer(2,1),
    activationLayer("ReLU")
])

test.getLayers()

0 Normalization
1 Weights
2 Biases
3 Activation
4 Weights
5 Biases
6 Activation
7 Weights
8 Biases
9 Activation


In [607]:
test.evaluateInput([1,2,3])

[0.8681872432382856]


[0.8681872432382856]